# 10 - CLI Reference Manual

> **When to use**: When you want to operate directly in the terminal without writing Python code.
>
> **Core concept**: The `sqlseed` CLI provides commands such as fill, preview, inspect, ai-suggest, and replay.

## Applicable Scenarios

- Quick fill test data → `sqlseed fill`
- Preview data without writing → `sqlseed preview`
- View column mapping strategies → `sqlseed inspect --show-mapping`
- AI generates config → `sqlseed ai-suggest`
- Replay snapshot → `sqlseed replay`

## What You Will Learn

- All CLI commands and options
- Output formatting
- Error handling

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| **→ 10** | **CLI Reference Manual** | **CLI** | **06** |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [1]:
import os
from pathlib import Path

from click.testing import CliRunner

import sqlseed
from sqlseed.cli.main import cli

os.environ["SQLSEED_LOG_LEVEL"] = "WARNING"




db_path = Path("../sqlseed_demo.db")
print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

runner = CliRunner()

def run_cli(*args):
    result = runner.invoke(cli, list(args))
    return result.output

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: ../sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| CLI Parsing | `src/sqlseed/cli/main.py` | `cli()` |

## 1. See It in Action — Fill Data with One Command

No need to write Python code; a single CLI command generates data:

```bash
sqlseed fill app.db --table users --count 10000 --seed 42
```

Below we demo all CLI commands and options.

## 2. Quick Fill: One Command

Simplest usage: specify the database path, table name, and row count.

In [2]:
# Fill 100 members via CLI (use --clear to avoid UNIQUE conflicts with existing data)
result = run_cli('fill', str(db_path), '-t', 'members', '-n', '100', '--seed', '42', '--clear')
print(result)

Generating members:   0%|          | 0/100 [00:00<?, ?it/s]

GenerationResult(table=members, count=100, elapsed=6.21s, speed=16.11 rows/s)



## 3. Preview Data: No Database Write

The `preview` command generates data without writing, suitable for verifying mapping results.

In [3]:
result = run_cli('preview', str(db_path), '-t', 'organizations', '-n', '3', '--seed', '42')
print(result)

                                          Preview: organizations (3 rows)                                          
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ org_code    ┃ name             ┃ parent_code ┃ description                         ┃ created_at                 ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ hbVrpoiVgRV │ Anthony Reilly   │ ORG-1819    │ Messages can be sent to and         │ 2020-02-01 23:17:15.234053 │
│             │                  │             │ received from ports, but these      │                            │
│             │                  │             │ messages must obey the so-called    │                            │
│             │                  │             │ "port protocol."                    │                            │
│ fLBcbfnoGM  │ Kai Day          │ ORG-0013    │ Haskell is a standardized,          │ 2004-12-04 21:47:57.571858 │
│             │                  │             │ general-purpose purely functional   │                            │
│             │                  │             │ programming language, with          │                            │
│             │                  │             │ non-strict semantics and strong     │                            │
│             │                  │             │ static typing.                      │                            │
│ JmTPSI      │ Cleveland Osborn │ ORG-0013    │ Erlang is known for its designs     │ 2002-10-14 01:01:05.229258 │
│             │                  │             │ that are well suited for systems.   │                            │
│             │                  │             │ The sequential subset of Erlang     │                            │
│             │                  │             │ supports eager evaluation, single   │                            │
│             │                  │             │ assignment, and dynamic typing.     │                            │
└─────────────┴──────────────────┴─────────────┴─────────────────────────────────────┴────────────────────────────┘

## 4. Inspect Schema and Column Mapping

`inspect` shows table structure; `--show-mapping` displays each column's mapping strategy.

In [4]:
result = run_cli('inspect', str(db_path), '-t', 'organizations', '--show-mapping')
print(result)

                                           Table: organizations (3 rows)                                           
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column       ┃ Type        ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                        ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ org_code     │ VARCHAR(16) │ ✗        │ ✓  │      │ string      │ {'min_length': 6, 'max_length': 12,           │
│              │             │          │    │      │             │ 'charset': 'alphanumeric'}                    │
│ name         │ VARCHAR(64) │ ✗        │    │      │ name        │ {}                                            │
│ parent_code  │ VARCHAR(16) │ ✓        │    │      │ foreign_key │ {'ref_table': 'organizations', 'ref_column':  │
│              │             │          │    │      │             │ 'org_code', 'strategy': 'random',             │
│              │             │          │    │      │             │ '_ref_values': ['ORG-0013', 'ORG-0433',       │
│              │             │          │    │      │             │ 'ORG-1819']}                                  │
│ description  │ TEXT        │ ✓        │    │      │ text        │ {'min_length': 100, 'max_length': 500}        │
│ is_active    │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ member_count │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ created_at   │ TEXT        │ ✓        │    │      │ datetime    │ {}                                            │
└──────────────┴─────────────┴──────────┴────┴──────┴─────────────┴───────────────────────────────────────────────┘

        Foreign Keys: organizations         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column      ┃ Ref Table     ┃ Ref Column ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ parent_code │ organizations │ org_code   │
└─────────────┴───────────────┴────────────┘

## 5. Generate Config Template: `init`

Auto-generates a YAML config based on the database schema; you can manually edit it and run with `fill -c`.

In [5]:
from pathlib import Path

init_path = Path('_cli_demo_config.yaml')
result = run_cli('init', str(init_path), '--db', str(db_path))
print(result)

# Show generated config
print(init_path.read_text()[:500])
init_path.unlink(missing_ok=True)

Configuration template saved to: _cli_demo_config.yaml

db_path: ../sqlseed_demo.db
provider: mimesis
locale: en_US
tables:
- name: organizations
  count: 1000
  batch_size: 5000
  columns: []
  clear_before: false
  seed: null
  transform: null
  enrich: false
- name: members
  count: 1000
  batch_size: 5000
  columns: []
  clear_before: false
  seed: null
  transform: null
  enrich: false
- name: projects
  count: 1000
  batch_size: 5000
  columns: []
  clear_before: false
  seed: null
  transform: null
  enrich: false
- name: tasks
  count: 1000
 


## 6. Fill from Config File: `fill -c`

Batch-fill using a YAML/JSON config file, supporting advanced features like multi-table, custom generators, and cross-table associations.

In [6]:
from sqlseed.config.loader import save_config
from sqlseed.config.models import GeneratorConfig, TableConfig

# Create a config file
config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name='tags', count=5, clear_before=True, seed=42),
    ]
)
config_path = Path('_cli_fill_config.yaml')
save_config(config, str(config_path))

# Fill from config
result = run_cli('fill', '-c', str(config_path), '--seed', '42')
print(result)

config_path.unlink(missing_ok=True)

Generating tags:   0%|          | 0/5 [00:00<?, ?it/s]

Loading config: _cli_fill_config.yaml (1 table(s))
GenerationResult(table=tags, count=5, elapsed=0.03s, speed=168.08 rows/s)



## 7. Error Handling and Exit Codes

CLI commands return a non-zero exit code to indicate an error.

In [7]:
# Missing required arguments
result = runner.invoke(cli, ['fill', str(db_path)])
print(f'Exit code: {result.exit_code}')
print(f'Error: {result.output.strip()[:200]}')

Exit code: 2
Error: Usage: cli fill [OPTIONS] [DB_PATH]
Try 'cli fill --help' for help.

Error: --count is required when not using --config. Use -n <number> to specify the number of rows to generate.


## Summary

| Command | Purpose | Writes to DB |
|------|------|:----------:|
| `fill` | Fill data | ✅ |
| `preview` | Preview without writing | ❌ |
| `inspect` | View schema | ❌ |
| `init` | Generate config template | ❌ |
| `fill -c` | Fill from config | ✅ |
| `replay` | Replay snapshot | ✅ |
| `ai-suggest` | AI generates config | ❌ |

**Next**: [11-utilities.ipynb](11-utilities.ipynb) — Utilities Reference

In [8]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
